In [1]:
%load_ext autoreload
%autoreload 2

In [21]:
import sys
import os
import copy
import numpy as np
from pathlib import Path
from omegaconf import OmegaConf
from tqdm import tqdm
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import time
from datetime import datetime

import pandas as pd
from IPython.display import display

# Add the project root to the path
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [4]:
root_dir = Path(module_path) / Path('aa_experiments')

In [10]:
exp_dir = root_dir / Path('cfo_final')
df_cfo = pd.read_csv(exp_dir / Path('eval_xtb_full.csv'))
df_cfo

,index,seed,step,graph_idx,valid,invalid_reason,simple_valid,simple_invalid_reason,qed,logp,...,atom_count_F,atom_count_P,atom_count_S,atom_count_Cl,atom_count_Br,atom_count_I,pred_reward,pred_constraint,true_reward,true_constraint
0,K,0,6,0,False,sanitize_fail,0.0,sanitize_fail,NaN,NaN,...,0,0,0,0,0,0,4.007875,-64.015160,4.894,-65.160135
1,K,0,6,1,False,disconnected,1.0,NaN,NaN,NaN,...,0,0,0,0,0,0,14.548645,-86.924767,3.582,-84.088228
2,K,0,6,2,False,sanitize_fail,0.0,sanitize_fail,NaN,NaN,...,0,0,0,0,0,0,4.053035,-71.270912,8.557,-73.142625
3,K,0,6,3,False,disconnected,0.0,sanitize_fail,NaN,NaN,...,0,0,0,0,0,0,6.281954,-76.666374,17.307,-75.006730
4,K,0,6,4,False,sanitize_fail,0.0,sanitize_fail,NaN,NaN,...,0,0,0,0,0,0,10.950938,-78.729843,12.268,-80.494196
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,K,9,6,195,False,disconnected,0.0,sanitize_fail,NaN,NaN,...,0,0,0,0,0,0,5.547248,-77.878151,5.759,-74.665193
1996,K,9,6,196,False,disconnected,1.0,NaN,NaN,NaN,...,0,0,0,0,0,0,5.253828,-74.495644,3.841,-76.947194
1997,K,9,6,197,False,disconnected,0.0,sanitize_fail,NaN,NaN,...,0,0,0,0,0,0,11.179401,-88.030693,18.007,-86.470707
1998,K,9,6,198,False,disconnected,0.0,sanitize_fail,NaN,NaN,...,0,0,0,0,0,0,5.802702,-60.228931,3.158,-61.044086


In [15]:
exp_dir = root_dir / Path('am_final')
df_am = pd.read_csv(exp_dir / Path('eval_xtb_full.csv'))
df_am

,index,seed,step,graph_idx,valid,invalid_reason,simple_valid,simple_invalid_reason,qed,logp,...,atom_count_F,atom_count_P,atom_count_S,atom_count_Cl,atom_count_Br,atom_count_I,pred_reward,pred_constraint,true_reward,true_constraint
0,K,0,1,0,False,sanitize_fail,0.0,sanitize_fail,NaN,NaN,...,0,0,0,0,0,0,4.344288,-65.128670,3.804,-66.979702
1,K,0,1,1,False,disconnected,0.0,sanitize_fail,NaN,NaN,...,0,0,0,0,0,0,10.868814,-75.801949,18.802,-77.475951
2,K,0,1,2,False,sanitize_fail,0.0,sanitize_fail,NaN,NaN,...,0,0,0,0,0,0,9.253829,-73.174126,13.086,-77.892098
3,K,0,1,3,False,sanitize_fail,0.0,sanitize_fail,NaN,NaN,...,0,0,0,0,0,0,8.796417,-78.943153,21.758,-85.418230
4,K,0,1,4,False,sanitize_fail,0.0,sanitize_fail,NaN,NaN,...,0,0,0,0,0,0,11.943373,-81.507393,16.594,-83.261934
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,K,9,1,195,False,disconnected,0.0,sanitize_fail,NaN,NaN,...,0,0,0,0,0,0,6.525631,-73.348770,5.010,-77.754476
1996,K,9,1,196,False,disconnected,1.0,NaN,NaN,NaN,...,0,0,0,0,0,0,4.922368,-75.539490,4.057,-76.980958
1997,K,9,1,197,False,sanitize_fail,0.0,sanitize_fail,NaN,NaN,...,0,0,0,0,0,0,7.815562,-78.274940,6.346,-78.948122
1998,K,9,1,198,False,sanitize_fail,0.0,sanitize_fail,NaN,NaN,...,0,0,0,0,0,0,5.274614,-62.473118,20.039,-65.464898


In [28]:
def last_step_stats_table(
    df: pd.DataFrame,
    *,
    step_col="step",
    seed_col="seed",
    sample_col="graph_idx",
    valid_col="valid",
    only_valid: bool = False,
    mode: str = "per_seed",   # "per_seed" | "pooled"
    exclude_cols=None,
    round_to: int = 3,
):
    """
    Statistics at the LAST step only.

    mode="per_seed":
        samples -> seed means -> CI across seeds

    mode="pooled":
        all samples treated equally -> CI across samples
    """

    if mode not in {"per_seed", "pooled"}:
        raise ValueError("mode must be 'per_seed' or 'pooled'")

    if exclude_cols is None:
        exclude_cols = {
            "index",
            seed_col,
            step_col,
            sample_col,
            "invalid_reason",
            "simple_invalid_reason",
            "murcko_scaffold_smiles",
        }

    # --- last step ---
    last_step = df[step_col].max()
    d = df[df[step_col] == last_step].copy()

    # --- optional valid-only filter ---
    if only_valid:
        d = d[d[valid_col] == True]

    # --- numeric columns INCLUDING valid ---
    numeric_cols = (
        d.select_dtypes(include=[np.number])
         .columns
         .difference(exclude_cols)
         .tolist()
    )

    rows = []

    if mode == "per_seed":
        # ---- per-seed means over samples ----
        per_seed = d.groupby(seed_col)[numeric_cols].mean()

        for col in numeric_cols:
            vals = per_seed[col].dropna()
            mean = vals.mean()
            sem = vals.sem()
            ci_lo = mean - 1.96 * sem
            ci_hi = mean + 1.96 * sem

            rows.append((col, mean, ci_lo, ci_hi))

    else:  # mode == "pooled"
        for col in numeric_cols:
            vals = d[col].dropna()
            mean = vals.mean()
            sem = vals.sem()
            ci_lo = mean - 1.96 * sem
            ci_hi = mean + 1.96 * sem

            rows.append((col, mean, ci_lo, ci_hi))

    # --- assemble table ---
    stats = pd.DataFrame(
        rows, columns=["metric", "mean", "ci_lo", "ci_hi"]
    )

    if round_to is not None:
        stats = stats.round(round_to)

    stats["mean ± 95% CI"] = (
        stats["mean"].astype(str)
        + " [" + stats["ci_lo"].astype(str)
        + ", " + stats["ci_hi"].astype(str) + "]"
    )

    display(stats)
    return stats


In [33]:
cfo_stats = last_step_stats_table(df_cfo, mode="per_seed")

,metric,mean,ci_lo,ci_hi,mean ± 95% CI
0,atom_count_Br,0.001,-0.000,0.002,"0.001 [-0.0, 0.002]"
1,atom_count_C,1.086,0.884,1.287,"1.086 [0.884, 1.287]"
2,atom_count_Cl,0.002,-0.000,0.004,"0.002 [-0.0, 0.004]"
3,atom_count_F,0.005,0.001,0.009,"0.005 [0.001, 0.009]"
4,atom_count_H,1.386,1.095,1.678,"1.386 [1.095, 1.678]"
5,atom_count_I,0.000,0.000,0.000,"0.0 [0.0, 0.0]"
6,atom_count_N,0.397,0.307,0.487,"0.397 [0.307, 0.487]"
7,atom_count_O,0.348,0.268,0.428,"0.348 [0.268, 0.428]"
8,atom_count_P,0.000,-0.000,0.001,"0.0 [-0.0, 0.001]"
9,atom_count_S,0.015,0.008,0.021,"0.015 [0.008, 0.021]"


In [ ]:
am_stats = last_step_stats_table(df_am, mode="per_seed")

,metric,mean,ci_lo,ci_hi,mean ± 95% CI
0,atom_count_Br,0.002,0.000,0.003,"0.002 [0.0, 0.003]"
1,atom_count_C,0.919,0.635,1.202,"0.919 [0.635, 1.202]"
2,atom_count_Cl,0.002,0.000,0.004,"0.002 [0.0, 0.004]"
3,atom_count_F,0.006,0.001,0.010,"0.006 [0.001, 0.01]"
4,atom_count_H,1.190,0.834,1.545,"1.19 [0.834, 1.545]"
5,atom_count_I,0.000,0.000,0.000,"0.0 [0.0, 0.0]"
6,atom_count_N,0.397,0.270,0.524,"0.397 [0.27, 0.524]"
7,atom_count_O,0.262,0.171,0.354,"0.262 [0.171, 0.354]"
8,atom_count_P,0.000,-0.000,0.001,"0.0 [-0.0, 0.001]"
9,atom_count_S,0.013,0.007,0.019,"0.013 [0.007, 0.019]"


In [ ]:
cfo_stats_pooled = last_step_stats_table(df_cfo, mode="pooled")

,metric,mean,ci_lo,ci_hi,mean ± 95% CI
0,atom_count_Br,0.001,-0.000,0.002,"0.001 [-0.0, 0.002]"
1,atom_count_C,1.086,0.913,1.258,"1.086 [0.913, 1.258]"
2,atom_count_Cl,0.002,0.000,0.004,"0.002 [0.0, 0.004]"
3,atom_count_F,0.005,0.000,0.010,"0.005 [0.0, 0.01]"
4,atom_count_H,1.386,1.166,1.607,"1.386 [1.166, 1.607]"
5,atom_count_I,0.000,0.000,0.000,"0.0 [0.0, 0.0]"
6,atom_count_N,0.397,0.330,0.464,"0.397 [0.33, 0.464]"
7,atom_count_O,0.348,0.288,0.408,"0.348 [0.288, 0.408]"
8,atom_count_P,0.000,-0.000,0.001,"0.0 [-0.0, 0.001]"
9,atom_count_S,0.014,0.009,0.020,"0.014 [0.009, 0.02]"


In [ ]:
am_stats_pooled = last_step_stats_table(df_am, mode="pooled")

,metric,mean,ci_lo,ci_hi,mean ± 95% CI
0,atom_count_Br,0.002,-0.000,0.003,"0.002 [-0.0, 0.003]"
1,atom_count_C,0.918,0.760,1.077,"0.918 [0.76, 1.077]"
2,atom_count_Cl,0.002,0.000,0.004,"0.002 [0.0, 0.004]"
3,atom_count_F,0.006,0.001,0.010,"0.006 [0.001, 0.01]"
4,atom_count_H,1.190,0.985,1.394,"1.19 [0.985, 1.394]"
5,atom_count_I,0.000,0.000,0.000,"0.0 [0.0, 0.0]"
6,atom_count_N,0.397,0.324,0.470,"0.397 [0.324, 0.47]"
7,atom_count_O,0.262,0.212,0.313,"0.262 [0.212, 0.313]"
8,atom_count_P,0.000,-0.000,0.001,"0.0 [-0.0, 0.001]"
9,atom_count_S,0.013,0.008,0.018,"0.013 [0.008, 0.018]"


In [34]:
def last_step_stats_table(
    df: pd.DataFrame,
    *,
    step_col="step",
    seed_col="seed",
    sample_col="graph_idx",
    valid_col="valid",
    simple_valid_col="simple_valid",
    only_valid: bool = False,     # applies ONLY to non-validity metrics
    mode: str = "per_seed",       # "per_seed" | "pooled"
    exclude_cols=None,
    round_to: int = 3,
):
    if mode not in {"per_seed", "pooled"}:
        raise ValueError("mode must be 'per_seed' or 'pooled'")

    if exclude_cols is None:
        exclude_cols = {
            "index", seed_col, step_col, sample_col,
            "invalid_reason", "simple_invalid_reason",
            "murcko_scaffold_smiles",
        }

    # --- last step slice ---
    last_step = df[step_col].max()
    d_last = df[df[step_col] == last_step].copy()

    # -------------------------
    # 1) VALIDITY FRACTIONS (always computed on ALL last-step samples)
    # -------------------------
    def _fraction_stats_from_bool_col(d, colname, out_metric_name):
        if colname not in d.columns:
            return None

        # Ensure boolean-ish -> float in {0,1}
        vals = d[colname].astype(float)

        if mode == "per_seed":
            per_seed_frac = vals.groupby(d[seed_col]).mean().dropna()
            mean = per_seed_frac.mean()
            sem = per_seed_frac.sem()
        else:  # pooled
            vals2 = vals.dropna()
            mean = vals2.mean()
            sem = vals2.sem()

        ci_lo = mean - 1.96 * sem
        ci_hi = mean + 1.96 * sem
        return (out_metric_name, mean, ci_lo, ci_hi)

    rows = []
    r = _fraction_stats_from_bool_col(d_last, valid_col, "valid_fraction")
    if r is not None:
        rows.append(r)

    r = _fraction_stats_from_bool_col(d_last, simple_valid_col, "simple_valid_fraction")
    if r is not None:
        rows.append(r)

    # Some helpful counts for sanity checking
    n_seeds = d_last[seed_col].nunique()
    n_samples_total = len(d_last)
    n_valid_total = int(d_last[valid_col].sum()) if valid_col in d_last.columns else None

    # -------------------------
    # 2) OTHER NUMERIC METRICS (optionally filter to only_valid)
    # -------------------------
    d_metrics = d_last.copy()
    if only_valid:
        d_metrics = d_metrics[d_metrics[valid_col] == True].copy()

    numeric_cols = (
        d_metrics.select_dtypes(include=[np.number])
        .columns.difference(exclude_cols)
        .tolist()
    )

    # Remove validity columns from numeric list to avoid duplicates / confusion
    for c in [valid_col, simple_valid_col]:
        if c in numeric_cols:
            numeric_cols.remove(c)

    if mode == "per_seed":
        per_seed = d_metrics.groupby(seed_col)[numeric_cols].mean()
        for col in numeric_cols:
            vals = per_seed[col].dropna()
            mean = vals.mean()
            sem = vals.sem()
            rows.append((col, mean, mean - 1.96 * sem, mean + 1.96 * sem))
    else:  # pooled
        for col in numeric_cols:
            vals = d_metrics[col].dropna()
            mean = vals.mean()
            sem = vals.sem()
            rows.append((col, mean, mean - 1.96 * sem, mean + 1.96 * sem))

    # -------------------------
    # 3) Assemble & display
    # -------------------------
    stats = pd.DataFrame(rows, columns=["metric", "mean", "ci_lo", "ci_hi"])

    if round_to is not None:
        stats[["mean", "ci_lo", "ci_hi"]] = stats[["mean", "ci_lo", "ci_hi"]].round(round_to)

    stats["mean ± 95% CI"] = (
        stats["mean"].astype(str)
        + " [" + stats["ci_lo"].astype(str)
        + ", " + stats["ci_hi"].astype(str) + "]"
    )

    # Put validity fractions at top
    validity_order = ["valid_fraction", "simple_valid_fraction"]
    stats["__rank"] = stats["metric"].apply(lambda m: validity_order.index(m) if m in validity_order else 999)
    stats = stats.sort_values(["__rank", "metric"]).drop(columns="__rank").reset_index(drop=True)

    # Show counts header + table
    print(f"Last step = {last_step} | mode = {mode} | only_valid(metrics) = {only_valid}")
    print(f"n_seeds = {n_seeds} | n_samples_total = {n_samples_total}" + (f" | n_valid_total = {n_valid_total}" if n_valid_total is not None else ""))
    display(stats)

    return stats


In [35]:
print("\nCFO Stats:")
cfo_stats = last_step_stats_table(df_cfo, mode="per_seed")
print("\nAM Stats:")
am_stats = last_step_stats_table(df_am, mode="per_seed")
print("\nCFO Pooled Stats:")
cfo_stats_pooled = last_step_stats_table(df_cfo, mode="pooled")
print("\nAM Pooled Stats:")
am_stats_pooled = last_step_stats_table(df_am, mode="pooled")


CFO Stats:
Last step = 6 | mode = per_seed | only_valid(metrics) = False
n_seeds = 10 | n_samples_total = 2000 | n_valid_total = 144


,metric,mean,ci_lo,ci_hi,mean ± 95% CI
0,valid_fraction,0.072,0.057,0.087,"0.072 [0.057, 0.087]"
1,simple_valid_fraction,0.130,0.110,0.151,"0.13 [0.11, 0.151]"
2,atom_count_Br,0.001,-0.000,0.002,"0.001 [-0.0, 0.002]"
3,atom_count_C,1.086,0.884,1.287,"1.086 [0.884, 1.287]"
4,atom_count_Cl,0.002,-0.000,0.004,"0.002 [-0.0, 0.004]"
5,atom_count_F,0.005,0.001,0.009,"0.005 [0.001, 0.009]"
6,atom_count_H,1.386,1.095,1.678,"1.386 [1.095, 1.678]"
7,atom_count_I,0.000,0.000,0.000,"0.0 [0.0, 0.0]"
8,atom_count_N,0.397,0.307,0.487,"0.397 [0.307, 0.487]"
9,atom_count_O,0.348,0.268,0.428,"0.348 [0.268, 0.428]"



AM Stats:
Last step = 1 | mode = per_seed | only_valid(metrics) = False
n_seeds = 10 | n_samples_total = 2000 | n_valid_total = 124


,metric,mean,ci_lo,ci_hi,mean ± 95% CI
0,valid_fraction,0.062,0.043,0.081,"0.062 [0.043, 0.081]"
1,simple_valid_fraction,0.086,0.068,0.104,"0.086 [0.068, 0.104]"
2,atom_count_Br,0.002,0.000,0.003,"0.002 [0.0, 0.003]"
3,atom_count_C,0.919,0.635,1.202,"0.919 [0.635, 1.202]"
4,atom_count_Cl,0.002,0.000,0.004,"0.002 [0.0, 0.004]"
5,atom_count_F,0.006,0.001,0.010,"0.006 [0.001, 0.01]"
6,atom_count_H,1.190,0.834,1.545,"1.19 [0.834, 1.545]"
7,atom_count_I,0.000,0.000,0.000,"0.0 [0.0, 0.0]"
8,atom_count_N,0.397,0.270,0.524,"0.397 [0.27, 0.524]"
9,atom_count_O,0.262,0.171,0.354,"0.262 [0.171, 0.354]"



CFO Pooled Stats:
Last step = 6 | mode = pooled | only_valid(metrics) = False
n_seeds = 10 | n_samples_total = 2000 | n_valid_total = 144


,metric,mean,ci_lo,ci_hi,mean ± 95% CI
0,valid_fraction,0.072,0.061,0.083,"0.072 [0.061, 0.083]"
1,simple_valid_fraction,0.130,0.116,0.145,"0.13 [0.116, 0.145]"
2,atom_count_Br,0.001,-0.000,0.002,"0.001 [-0.0, 0.002]"
3,atom_count_C,1.086,0.913,1.258,"1.086 [0.913, 1.258]"
4,atom_count_Cl,0.002,0.000,0.004,"0.002 [0.0, 0.004]"
5,atom_count_F,0.005,0.000,0.010,"0.005 [0.0, 0.01]"
6,atom_count_H,1.386,1.166,1.607,"1.386 [1.166, 1.607]"
7,atom_count_I,0.000,0.000,0.000,"0.0 [0.0, 0.0]"
8,atom_count_N,0.397,0.330,0.464,"0.397 [0.33, 0.464]"
9,atom_count_O,0.348,0.288,0.408,"0.348 [0.288, 0.408]"



AM Pooled Stats:
Last step = 1 | mode = pooled | only_valid(metrics) = False
n_seeds = 10 | n_samples_total = 2000 | n_valid_total = 124


,metric,mean,ci_lo,ci_hi,mean ± 95% CI
0,valid_fraction,0.062,0.051,0.073,"0.062 [0.051, 0.073]"
1,simple_valid_fraction,0.086,0.074,0.098,"0.086 [0.074, 0.098]"
2,atom_count_Br,0.002,-0.000,0.003,"0.002 [-0.0, 0.003]"
3,atom_count_C,0.918,0.760,1.077,"0.918 [0.76, 1.077]"
4,atom_count_Cl,0.002,0.000,0.004,"0.002 [0.0, 0.004]"
5,atom_count_F,0.006,0.001,0.010,"0.006 [0.001, 0.01]"
6,atom_count_H,1.190,0.985,1.394,"1.19 [0.985, 1.394]"
7,atom_count_I,0.000,0.000,0.000,"0.0 [0.0, 0.0]"
8,atom_count_N,0.397,0.324,0.470,"0.397 [0.324, 0.47]"
9,atom_count_O,0.262,0.212,0.313,"0.262 [0.212, 0.313]"
